# Syntethic data

## init

In [ ]:
%pip install .

In [1]:
import base64
import io
import pickle
import functools

from IPython.display import HTML, display
import jax.numpy as jnp
import jax
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import pandas as pd
import plotly.graph_objects as go

from hnmf_tr_optimizer.hnmf_optimizer import HNMFOptimizer, NewHNMFOptimizer, flatten, unflatten
from hnmf_tr_optimizer.clusts import result_analysis

from dls_model import observational_matrix, gen_bounds, get_init_params, clustering_preprocess, InitParamsGenerator
# from dls_lognormal import g1, scatter_vector, diffusion_coef

# plt.style.use('Solarize_Light2')

## generate data

In [2]:
exp = 5
noise_level = 1e-9
output_file = '5s_n9.pkl'

In [ ]:
if exp == 1:
    amp = jnp.array([1.])
    mu = jnp.array([15.])
    sig = jnp.array([4.])
elif exp == 2:
    amp = jnp.array([0.6, 1])
    amp = amp/jnp.sum(amp)
    mu = jnp.array([5., 25])
    sig = jnp.array([2., 5])
elif exp == 3:
    amp = jnp.array([0.6, 1, 0.4])
    amp = amp/jnp.sum(amp)
    mu = jnp.array([5., 15, 40])
    sig = jnp.array([2., 4, 8])
elif exp == 4:
    amp = jnp.array([0.4, 0.8, 0.6, 1])
    amp = amp/jnp.sum(amp)
    mu = jnp.array([10., 20, 30, 40])
    sig = jnp.array([2., 5, 3, 6])
elif exp == 5:
    amp = jnp.array([0.2, 0.4, 0.6, 0.3, 0.4])
    amp = amp/jnp.sum(amp)
    mu = jnp.array([10., 20, 30, 45, 70])
    sig = jnp.array([1., 1.5, 3, 2, 5])
    #### old ####
    # amp = jnp.array([0.2, 0.4, 0.6, 0.3, 0.4])
    # amp = amp/jnp.sum(amp)
    # mu = jnp.array([10., 20, 30, 40, 50])
    # sig = jnp.array([1., 4, 3, 2, 5])


# Kbt = 4.11e-21 # Joules
# vis = 0.00089 # viscosity of water at 25C
l = 633e-9 # He-Ne laser wavelength
n = 1.33 # water refractive index

theta = jnp.concatenate([jnp.arange(20., 50, 10), jnp.arange(50, 121, 5)]) # degrees
q = (4*jnp.pi*n/l)*jnp.sin(jnp.radians(theta/2))

t = jnp.linspace(1e-6, 1e-3, 100)
observations = observational_matrix(q, t, amp, mu, sig)


key = jax.random.key(1337)
key, subkey = jax.random.split(key)
noise = jax.random.normal(subkey, shape=observations.shape)*noise_level

observations = observations + noise

## run minimizations

In [ ]:

opt = HNMFOptimizer(
    model_fn=observational_matrix,
    param_generator=InitParamsGenerator(),
    bound_generator=gen_bounds,
    input_args = ('q', 't'),
    param_args=('amp', 'mu', 'sig'),
    constants = {},
    min_k=exp-1,
    max_k=exp+1,
    nsim=100
)

res = opt(
    (q, t),
    observations,
    opt_options={
        'fatol': 1e-13,
        'frtol': 0.0,
        'maxiter': 2000,
        'gatol': 1e-11
    }
)

res['real_sol'] = [(amp, mu, sig)]*len(res.index)

res.to_pickle(output_file)

Forclusts = clustering_preprocess(res)

report = Forclusts.groupby('num_sources', group_keys=False)[['num_sources', 'points', 'normF']].apply(lambda group: result_analysis(
    group['points'].sum(),
    group['normF'].mean(),
    observations.size,
    group['num_sources'].iloc[0]
))

print(report)

## visualizations from results

In [4]:
def normal_distribution(x, amplitude, mu, sigma):
    return amplitude * jnp.exp(-(x-mu)**2/(2*sigma**2))/jnp.sqrt(2*jnp.pi*sigma**2)

nnn = jax.vmap(normal_distribution, in_axes=(None, 0, 0, 0))
x_values = jnp.linspace(0.1, 100, 1000)


with open(output_file, 'rb') as f:
    res = pickle.load(f)

best = res[res['num_sources'] == exp].dropna().sort_values(by='fval')
sols = pd.DataFrame(best['sol'].to_list(), columns=['amp', 'mu', 'sig'])

selected = [0, 5, 10, 15, 20, 25]

In [ ]:
real_mass_distributions = nnn(x_values, amp, mu, sig)
real_full_dist = jnp.sum(real_mass_distributions, axis=0)


plt.clf()
num_plots = len(selected)+1
fig, axes = plt.subplots(num_plots, 3, figsize=(40, 5*num_plots))

ax = axes[0]
ax[0].set_title('real mass distribution')
ax[0].plot(x_values, real_mass_distributions.T, linestyle='--')
ax[0].plot(x_values, real_full_dist)

ax[1].set_title('observations')
ax[1].plot(t, observations.T)
ax[1].set_xscale('log')

ax[2].set_title('noise')
for th in range(len(theta)):
    ax[2].scatter(t, noise[th,:])
# ax[2].scatter(jnp.stack([t]*len(theta)).ravel(), noise.T.ravel())
ax[2].set_xscale('log')

for i, ax in enumerate(axes[1:]):
    mass_distributions = nnn(x_values, *sols.iloc[selected[i]])
    full_dist = jnp.sum(mass_distributions, axis=0)
    ax[0].set_title(f'mass distribution solution #{selected[i]+1}')
    ax[0].plot(x_values, mass_distributions.T, linestyle='--')
    ax[0].plot(x_values, full_dist)
    for m in mu.tolist():
        ax[0].axvline(m, color='black', linestyle='--', zorder=-1)
    for m in sols.iloc[selected[i]]['mu'].tolist():
        ax[0].axvline(m, color='black', linestyle=':', zorder=-1)

    pred_observations = observational_matrix(q, t, *sols.iloc[selected[i]])
    ax[1].set_title('predicted observations')
    ax[1].plot(t, pred_observations.T)
    ax[1].set_xscale('log')

    ax[2].set_title('error residuals')
    resid = pred_observations - observations
    for th in range(len(theta)):
        ax[2].scatter(t, resid[th,:])
    ax[2].set_xscale('log')

fig.tight_layout()

plt.show()

In [174]:
def plotly_3d_surface(data, x, y):
    # x, y = jnp.linspace(0, data.shape[0]), jnp.linspace(0, data.shape[1])
    fig = go.Figure(data=[go.Surface(z=data, x=x, y=y)])
    fig.update_layout(autosize=False)
    return fig
    # fig.show()
    # return fig.to_html(full_html=False)

plt3d = functools.partial(plotly_3d_surface, x=jnp.log10(t), y=theta)


In [ ]:
plt3d(observations).show()

In [ ]:
plt3d(pred_observations).show()

In [ ]:
plt3d(pred_observations-observations).show()

In [26]:
# real_mass_distributions = nnn(x_values, amp, mu, sig)
# real_full_dist = jnp.sum(real_mass_distributions, axis=0)


# plt.clf()
# num_plots = len(selected) + 1
# fig, axes = plt.subplots(num_plots, 1, figsize=(18, 7*num_plots))

# ax = axes[0]
# ax.set_title('real data')
# ax.plot(x_values, real_mass_distributions.T, linestyle='--')
# ax.plot(x_values, real_full_dist)

# for i, ax in enumerate(axes[1:]):
#     mass_distributions = nnn(x_values, *sols.iloc[selected[i]])
#     full_dist = jnp.sum(mass_distributions, axis=0)
#     ax.set_title(f'minimization result #{selected[i]+1}')
#     ax.plot(x_values, mass_distributions.T, linestyle='--')
#     ax.plot(x_values, full_dist)
#     for m in mu.tolist():
#         ax.axvline(m, color='black', linestyle='--', zorder=-1)
#     for m in sols.iloc[selected[i]]['mu'].tolist():
#         ax.axvline(m, color='black', linestyle=':', zorder=-1)

# fig.tight_layout()

# plt.show()

In [ ]:
best['init_vals'].iloc[0]

In [ ]:
### visualize optimization ###

opt = HNMFOptimizer(
    model_fn=observational_matrix,
    param_generator=InitParamsGenerator(),
    bound_generator=gen_bounds,
    input_args = ('q', 't'),
    param_args=('amp', 'mu', 'sig'),
    constants = {},
    min_k=exp-1,
    max_k=exp+1,
    nsim=100
)

optimizer = opt.setup_optimizer(exp, (q, t), observations, opt_options={
        'fatol': 1e-12,
        'frtol': 0,
        'maxiter': 2000,
        'gatol': 1e-10
    }
)

# flat_init, _ = opt.flatten(*opt.param_generator(exp))
flat_init, _ = flatten(*best['init_vals'].iloc[0])
res = optimizer.full_trace_minimize(flat_init)
print(f"finished minimization: {len(res)} steps")

df = pd.DataFrame(res)

df['x'] = df['x'].apply(lambda x: opt.unflatten(x, opt.num_source2shapes(exp)))

step_params = pd.DataFrame(df['x'].to_list(), columns=['amp', 'mu', 'sig'])

def normal_distribution(x, amplitude, mu, sigma):
    return amplitude * jnp.exp(-(x-mu)**2/(2*sigma**2))/jnp.sqrt(2*jnp.pi*sigma**2)

nnn = jax.vmap(normal_distribution, in_axes=(None, 0, 0, 0))
x_values = jnp.linspace(0.1, 100, 1000)

interval = 50
step_params = step_params.iloc[[0] + list(range(interval, len(step_params)-1, interval)) + [len(step_params)-1]]

step_params['mass_distributions'] = step_params.apply(lambda row: nnn(x_values, *row), axis=1)
step_params['full_dist'] = step_params['mass_distributions'].apply(lambda x: jnp.sum(x, axis=0))

In [ ]:
plt.ioff()
plt.clf()
fig = plt.figure(figsize=(12, 8), dpi=200)
ax = fig.add_subplot(1,1,1)
ax.set_title('minimization results')
lines1 = ax.plot(x_values, step_params['mass_distributions'].iloc[0].T, linestyle=':')
line2 = ax.plot(x_values, step_params['full_dist'].iloc[0])[0]

def update(frame):
    for i, line1 in enumerate(lines1):
        line1.set_ydata(step_params['mass_distributions'].iloc[frame].T[:, i])
    line2.set_ydata(step_params['full_dist'].iloc[frame])
    ax.set_title(f"step: {step_params.index[frame]}")
    return lines1, line2

fig.tight_layout()
ani = animation.FuncAnimation(fig=fig, func=update, frames=20, interval=150)

HTML(ani.to_jshtml())

In [ ]:
step_params['mu'].iloc[5]

In [ ]:
plt.ioff()
plt.clf()
fig, ax = plt.subplots(dpi=150)
im = ax.imshow(jnp.log(observations.T))
cbar = ax.figure.colorbar(im, ax=ax)
fig.tight_layout()


buf = io.BytesIO()
fig.savefig(buf, format="png")
buf.seek(0)
encoded = base64.b64encode(buf.read()).decode("utf-8")
buf.close()

# Create HTML image tag
html_img = f'<img src="data:image/png;base64,{encoded}" style="max-width:100%;">'

# Display in Jupyter Notebook
HTML(html_img)


In [ ]:
# set up the figure and Axes
plt.ioff()
plt.clf()
fig = plt.figure(figsize=(8, 6), dpi=200)
ax1 = fig.add_subplot(111, projection='3d')

# fake data
_x = jnp.arange(observations.shape[0])
_y = jnp.arange(observations.shape[1])
_xx, _yy = jnp.meshgrid(_x, _y)
x, y = _xx.ravel(), _yy.ravel()

top = 2-jnp.log(observations.T).ravel()
bottom = jnp.zeros_like(top)
width = depth = 1

ax1.bar3d(x, y, bottom, width, depth, top, shade=True)
ax1.set_title('Shaded')

plt.show()

# Experimental data

## view

wavevector $q$ is a function of the angle $\theta$ is given by
$$q = \frac{4\pi n}{\lambda_0} \sin(\frac{\theta}{2})$$

Diffusion coefficient $D$ is a function of the size/radius $r$
$$D = \frac{k_B T}{6 \pi \eta r} $$

let $\Gamma(r) = D q^2$ be the decay rate corresponding to the particular radius (let angle be fixed for now). Now let $P(r)$ be our distribution of paticle radiuses. putting this together we get the definition for the autocorrelation function $g_1$:

$$g_1(\tau) = \int_{0}^{\infty} P(r) e^{-\Gamma(r)\tau} \,dr$$

where $\tau$ is the time delay. Notice that if we let $G(\Gamma)$ be the distribution of decay rates $\Gamma$, we could swap out the size distribution $P(r)$ by using $P(r)dr = G(\Gamma)d\Gamma$, allowing us to express the autocorrelation function as the laplace transform of the distribution of $\Gamma \propto \frac{1}{r}$

$$g_1(\tau) = \int_{0}^{\infty} G(\Gamma) e^{-\Gamma\tau} \,d\Gamma$$

Lastly, it can also be expressed using the distribution of diffusion coefficients $A(D)$

$$g_1(\tau) = \int_{0}^{\infty} A(D) e^{-q^2 D\tau} \,dD$$


CONTIN's approach here is to find a discrete distribution of $P(r)$, which allows us to convert the integral into a sum. This allows for modeling a single source as a point distribution and modeling multiple sources as a sume of point distributions.

Our approach is to instead model each source as a normal curve:

$$P_i(r) = \frac{\alpha}{\sqrt{2\pi\sigma^2}} e^{-\frac{(r - \mu)^2}{2\sigma^2}}$$

and $P(r) = \sum P_i(r)$ . Where $\alpha$ is the amplitude, $\mu$ is the mean particle radius, $\sigma$ is the standard deviation. To get $A(D)$, we can rearrange the definition of $D$ to get $r$ as a function of $D$, and we can say $A(D) = P(r(D))$ and then translate 

<!-- $$A_i(D)  = \frac{\alpha}{\sqrt{2\pi\sigma^2}} e^{-\frac{(x - \mu)^2}{2\sigma^2}}$$ -->

$$g_1(\tau) = \int_0^{\infty} \frac{\alpha}{\sqrt{2\pi\sigma^2}} e^{-\frac{(r - \mu)^2}{2\sigma^2}} e^{-q^2 \frac{k_B T}{6 \pi \eta r} \tau} \,dr$$

$$
g_1(\tau)=\frac{\alpha}{2}\,\exp\Bigl[-\frac{q^2k_BT\,\tau}{6\pi\eta\,\mu}\Bigr]
\left\{
\text{erfc}\!\Bigl[\frac{\mu-\sqrt{\displaystyle\frac{q^2k_BT\,\tau}{6\pi\eta}}}{\sqrt{2}\,\sigma}\Bigr]
+\text{erfc}\!\Bigl[\frac{\mu+\sqrt{\displaystyle\frac{q^2k_BT\,\tau}{6\pi\eta}}}{\sqrt{2}\,\sigma}\Bigr]
\right\}
$$


In [106]:
### new g1 formula ###
def scatter_vector(theta, lambda_0=633e-9, n=1.33):
    """
    theta - scatter angle
    lambda_0 - lsder wavelength in meters
    n - refractive index of water
    """
    return (4 * jnp.pi * n/lambda_0) * jnp.sin(theta / 2)

def diffusion_coef(r, k_B=138.e-23, T=298.15, eta=0.00089, n=1.33):
    """
    r - particle radius
    k_b - Boltzmann constant (J/K)
    T - Temperature (K)
    eta - Viscosity of water at room temerature (Pa*s)
    n - refractive index of water
    """
    return k_B * T / (6 * jnp.pi * eta * r)

def g1_single(mu, sigma, alpha, theta, tau):
    """
    Parameters:
      tau      - time lag (scalar or array)
      alpha    - amplitude scaling factor
      mu       - mean particle radius (m)
      sigma    - standard deviation of the radius (m)
      theta    - scattering angle (radians)
      lambda_0 - laser wavelength (m, default 633e-9)
      n        - refractive index of the medium (default 1.33)
      k_B      - Boltzmann constant (default 138e-23 J/K)
      T        - temperature (default 298.15 K)
      eta      - viscosity (default 0.00089 Pa*s)

    Returns:
      g1: the computed field autocorrelation function.
    """
    q = scatter_vector(theta)
    D_mu = diffusion_coef(mu)
    A = q**2 * mu * D_mu * tau
    exp_term = jnp.exp(-A / mu)
    erfc_arg1 = (mu - jnp.sqrt(A)) / (jnp.sqrt(2) * sigma)
    erfc_arg2 = (mu + jnp.sqrt(A)) / (jnp.sqrt(2) * sigma)
    return (alpha / 2) * exp_term * (jax.scipy.special.erfc(erfc_arg1) + jax.scipy.special.erfc(erfc_arg2))


g1_time = jax.vmap(
    g1_single,
    in_axes = (None, None, None, None, 0)
)

g1_k = jax.vmap(
    g1_single,
    in_axes=(0, 0, 0, None, None)
)

def g1_k_sum(r, sig, a, q, tau):
    return jnp.sum(g1_k(r, sig, a, q, tau), axis=0) # shape (k, t) -> (t,)

g1_theta = jax.vmap(
    g1_k_sum,
    in_axes = (None, None, None, 0, None)
)

# def g1(r, sig, a, theta, tau):
def g1(theta, tau, a, r, sig):
    """
    r - radius/mean size
    sig - std of radius/size
    a - amplitude of size distribution
    theta - angle in radians
    tau - time in seconds (milliseconds?)
    """
    # shitty wrapper to make inspect.getfullargspec work within HNMFOptimizer
    return g1_theta(r, sig, a, theta, tau)


In [107]:
# def scatter_vector(theta, lambda_0=633e-9, n=1.33):
#     """
#     theta - scatter angle
#     lambda_0 - lsder wavelength in meters
#     n - refractive index of water
#     """
#     return (4 * jnp.pi * n/lambda_0) * jnp.sin(theta / 2)

# def diffusion_coef(r, k_B=138.e-23, T=298.15, eta=0.00089, n=1.33):
#     """
#     r - particle radius
#     k_b - Boltzmann constant (J/K)
#     T - Temperature (K)
#     eta - Viscosity of water at room temerature (Pa*s)
#     n - refractive index of water
#     """
#     return k_B * T / (6 * jnp.pi * eta * r)

# def g1_single(r, sig, amp, q, tau):
#     """
#     r - particle radius
#     a - amplitude for particle size dist.
#     q - wave vector based on scatter angle
#     tau - time delay
#     """
#     D = diffusion_coef(1/sig)*1e6 # scaling by 1e6
#     # D = (2.45e-11)*(sig)
#     a = r/sig
#     gamma = D * q**2
#     x = (-(r/sig) + gamma*tau)/jnp.sqrt(2)
#     y = jnp.where(
#         jnp.greater_equal(x, 5),
#         1/(x*jnp.sqrt(jnp.pi)),
#         jax.scipy.special.erfc(x)*jnp.exp(x**2)
#     )
#     e = (amp/2) * jnp.exp(-(a**2/2))
#     return e*y

# g1_time = jax.vmap(
#     g1_single,
#     in_axes = (None, None, None, None, 0)
# )

# g1_k = jax.vmap(
#     g1_single,
#     in_axes=(0, 0, 0, None, None)
# )

# def g1_k_sum(r, sig, a, q, tau):
#     return jnp.sum(g1_k(r, sig, a, q, tau), axis=0) # shape (k, t) -> (t,)

# g1_theta = jax.vmap(
#     g1_k_sum,
#     in_axes = (None, None, None, 0, None)
# )

# def g1(r, sig, a, theta, tau):
#     """
#     r - radius/mean size
#     sig - std of radius/size
#     a - amplitude of size distribution
#     theta - angle in radians
#     tau - time in seconds (milliseconds?)
#     """
#     q = scatter_vector(theta)
#     # shitty wrapper to make inspect.getfullargspec work within HNMFOptimizer
#     return g1_theta(r, sig, a, q, tau)

In [ ]:
import 

In [122]:
df = pd.read_csv("~/repos/DLS/Experimental_data_083122/stock_100nm.csv", delimiter='\t', header=None)
df = df.iloc[1:] # remove strange first point
d = jnp.array(df.to_numpy())

t = d[:, 0] * 1e-3
theta = jnp.arange(30., 151, 5)

exp_obs = d[:, 1:].T


In [ ]:
plt.clf()
fig, ax = plt.subplots(1, 1, figsize=(18, 7))

ax.set_title('observations')
ax.plot(t, exp_obs.T)
ax.set_xscale('log')

fig.tight_layout()

plt.show()

In [125]:
# obs1 = g1(
#     r=jnp.array([1., 2.]),
#     sig=jnp.array([.5, 1.]),
#     a=jnp.array([0.5, 0.5]),
#     theta=jnp.radians(theta),
#     tau=t
# )

# plt.clf()
# fig, ax = plt.subplots(1, 1, figsize=(18, 7))

# ax.set_title('observations')
# ax.plot(t, obs1.T)
# ax.set_xscale('log')

# fig.tight_layout()

# plt.show()

In [16]:
def plotly_3d_surface(data, x, y):
    # x, y = jnp.linspace(0, data.shape[0]), jnp.linspace(0, data.shape[1])
    fig = go.Figure(data=[go.Surface(z=data, x=x, y=y)])
    fig.update_layout(autosize=False)
    return fig
    # fig.show()
    # return fig.to_html(full_html=False)

plt3d = functools.partial(plotly_3d_surface, x=jnp.log10(t), y=theta)


In [ ]:
plt3d(exp_obs).show()

In [ ]:
div = jax.vmap(lambda x: x/x[0])
norm_exp_obs = div(exp_obs)
plt3d(norm_exp_obs).show()

## minimize on experimental data

In [128]:
### filter within a time window
index = jnp.logical_and(t > 1e-5, t < 1e-1)
t_window = t[index]
exp_obs_window = exp_obs[:, index]

In [ ]:
### visualize optimization ###

k = 1

opt = HNMFOptimizer(
    model_fn=observational_matrix,
    param_generator=InitParamsGenerator(),
    bound_generator=gen_bounds,
    input_args = ('q', 't'),
    param_args=('amp', 'mu', 'sig'),
    constants = {},
    min_k=1,
    max_k=3,
    nsim=100
)

q = scatter_vector(theta)

optimizer = opt.setup_optimizer(k, (q, t_window), exp_obs_window, opt_options={
        'fatol': 1e-12,
        'frtol': 0,
        'maxiter': 2000,
        'gatol': 1e-10
    }
)

# opt = HNMFOptimizer(
#     model_fn=g1,
#     param_generator=InitParamsGenerator(),
#     bound_generator=gen_bounds,
#     input_args = ('theta', 'tau'),
#     param_args=('a', 'r', 'sig'),
#     constants = {},
#     min_k=1,
#     max_k=3,
#     nsim=100
# )

# div = jax.vmap(lambda x: x/x[0])
# norm_exp_obs = div(exp_obs_window)

# optimizer = opt.setup_optimizer(k, (theta, t_window), norm_exp_obs, opt_options={
#         'fatol': 1e-12,
#         'frtol': 0,
#         'maxiter': 2000,
#         'gatol': 1e-10
#     }
# )

flat_init, _ = opt.flatten(*opt.param_generator(k))
res = optimizer.full_trace_minimize(flat_init)
print(f"finished minimization: {len(res)} steps")

df = pd.DataFrame(res)

df['x'] = df['x'].apply(lambda x: opt.unflatten(x, opt.num_source2shapes(k)))

step_params = pd.DataFrame(df['x'].to_list(), columns=['amp', 'mu', 'sig'])

In [165]:
sol = step_params.iloc[-1]
pred_obs = observational_matrix(q, t, sol['amp'], sol['mu'], sol['sig'])
# pred_obs = g1(theta, t, *sol)


In [ ]:
sol

In [ ]:
plt3d(div(exp_obs)).show()

In [ ]:
plt3d(pred_obs).show()

In [ ]:
plt3d(pred_obs - div(exp_obs)).show()

In [ ]:
plt.clf()
fig, ax = plt.subplots(1, 1, figsize=(18, 7))

ax.set_title('residuals')
ax.plot(t, (pred_obs - exp_obs).T)
ax.set_xscale('log')

fig.tight_layout()

plt.show()